# Common Libraries

In [1]:
import os, mujoco
import numpy as np
from mink import Configuration, SE3, SO3, FrameTask, solve_ik
import matplotlib.pyplot as plt
import mediapy as media

# Model specification

In [2]:
dir_path = "/Users/seojin/mink/examples"
model_path = os.path.join(dir_path, "franka_emika_panda/mjx_scene.xml")
model = mujoco.MjModel.from_xml_path(model_path)

configuration = Configuration(model)

# Update the initial value of the configuration from the "home" keyframe.
configuration.update_from_keyframe("home")

# Defining a Target Pose

mink represents rigid body transformations using SE3 (poses) and SO3 (rotations). These classes support composition via @ and provide convenient constructors. 

In [33]:
# Get current end-effector pose.
ee_pose = configuration.get_transform_frame_to_world("attachment_site", "site")

# Define target (Translation + Rotation)
translation = np.array([0.0, -0.4, -0.2])
translated_ee_pose = SE3.from_translation(translation) @ ee_pose  # World frame
rotated_ee_pose = translated_ee_pose @ SE3.from_rotation(SO3.from_y_radians(-np.pi / 2)) # Local frame
target = rotated_ee_pose

- Left-multiplying applies a transformation in the **world frame**. The translation moves the pose along world axes. 
- Right-multiplying applies it in the **local frame**. The rotation is therefore about the gripper’s own y-axis, not the world’s.

This produces a target that is offset from the current pose and rotated 90°. Decoupling translation and rotation makes each component easier to reason about independently.

# Creating a Task

A FrameTask drives a frame toward a target pose.

Setting
- position_cost, orientation_cost: to non-zero values creates a full 6-DOF pose task
- orientation_cost = 0.0: for a position-only task where the end-effector can rotate freely
- convergence speed: The gain parameter sets the desired rate at which the task error should decay. With gain $g \in [0, 1]$, the error after steps is approximately: $$ e_n = e_0 \cdot (1-g)^n $$

To reach 1% of the initial error after n frames, solve for: $$ g = 1 - (0.01)^{1/n} $$

For example, to reduce error to 1% of initial over 2 seconds at 60 fps (120 frames):

Lower gain values yield smoother, more gradual convergence. The default gain=1.0 produces maximum convergence rate.

This formula assumes first-order error dynamics, which holds when the Jacobian is well-conditioned and the target is reachable.

In [45]:
duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
gain = 1.0 - 0.01 ** (1.0 / n_frames)  # ≈ 0.038

# Set target
task = FrameTask(
    frame_name="attachment_site",
    frame_type="site",
    position_cost=1.0,
    orientation_cost=1.0,
    gain=gain,
)
task.set_target(target)

# Update configuration
configuration.data.mocap_pos[0] = target.wxyz_xyz[4:]
configuration.data.mocap_quat[0] = target.wxyz_xyz[:4]

Lower gain values yield smoother, more gradual convergence. The default gain=1.0 produces maximum convergence rate.

# The IK Loop

At each step, solve_ik() computes a joint velocity intended to reduce task error:

- dt: the time between successive calls to solve_ik
    - Internall,y the QP solves for a configuration displacement $ \Delta q $ and enforces velocity limits as $|| \Delta q || \leq V_{max} \cdot dt$. These constraints scale correctly with dt, so limit enforcement is independent of the chosen timestep.
    - The main practical effect of dt is on step size: larger values produce larger configuration updates, which can degrade accuracy when the first-order linearization becomes linvalid over that step.
    - Note that dt need not equal the robot's servo rate; IK may run at a different frequency, with a lower-level controller interpolating between updates.

integrate_inplace() updates joint positions via q <- q + vel * dt, using proper quaternion intergration for ball and free joints

In [46]:
renderer = mujoco.Renderer(configuration.model, height=480, width=640)

frames = []
dt = 1.0 / fps
solver_name = "quadprog" 

for _ in range(n_frames):
    # Perform inverse kinematics
    vel = solve_ik(configuration, [task], dt, solver=solver_name)
    configuration.integrate_inplace(vel, dt)
    
    # Update the physical state of the simulation
    mujoco.mj_forward(configuration.model, configuration.data)
    
    # Save the current frame for video
    renderer.update_scene(configuration.data)
    pixels = renderer.render()
    frames.append(pixels)

media.show_video(frames, fps=fps, loop = False)

This browser does not support the video tag.


The dt parameter is the time between successive calls to solve_ik. Internally, the QP solves for a configuration displacement 
 and enforces velocity limits as . These constraints scale correctly with dt, so limit enforcement is independent of the chosen timestep. The main practical effect of dt is on step size: larger values produce larger configuration updates, which can degrade accuracy when the first-order linearization becomes invalid over that step. Note that dt need not equal the robot’s servo rate; IK may run at a different frequency, with a lower-level controller interpolating between updates.

integrate_inplace() updates joint positions via q ← q + vel * dt, using proper quaternion integration for ball and free joints.